In [9]:
import pandas as pd
import os
from sqlalchemy import create_engine
import logging
import time

# Setup logging
logging.basicConfig(
    filename="logs/ingestion_db.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

# Create SQLite database engine
engine = create_engine('sqlite:///inventory.db')

def ingest_db(df, table_name, engine):
    """This function ingests the dataframe into database table"""
    df.to_sql(table_name, con=engine, if_exists='replace', index=False)
    logging.info(f"Table {table_name} created/updated in DB")

def load_raw_data():
    """This function loads all CSVs as dataframe and ingests them into DB"""
    start = time.time()
    
    # Path to your CSV folder (relative so it works anywhere)
    data_dir = os.path.join(os.getcwd(), "data", "data")  
    
    if not os.path.exists(data_dir):
        logging.error(f"Data folder not found: {data_dir}")
        return
    
    for file in os.listdir(data_dir):
        if file.endswith(".csv"):
            file_path = os.path.join(data_dir, file)
            try:
                df = pd.read_csv(file_path)
                logging.info(f'Ingesting {file} into DB')
                ingest_db(df, file[:-4], engine)  # remove ".csv" for table name
            except Exception as e:
                logging.error(f"Error ingesting {file}: {e}")
    
    end = time.time()
    total_time = (end - start) / 60
    logging.info('------------- Ingestion Complete -------------')
    logging.info(f'Total Time Taken: {total_time:.2f} minutes')

if __name__ == "__main__":
    load_raw_data()
